# Module 1 instructor demo: a helpdesk ticket

Instructor only. Used live in the "Instructor demo" part of **`lectures/dsca-module-01.html`**, slides "Step 3: choose test sentences", "Step 4: run it and read the result" and "Step 5: choose what happens when it fails".

Nobody types anything in class. You run each cell and read the result aloud using the same three checks as the slide:

1. **Valid shape?** Is the reply JSON that follows the schema?
2. **Right values?** Are the values correct for this sentence?
3. **Null when unsure?** When the sentence does not say something, did the model use `null` and not guess?

Run it once before class with a real connection and save it with the output kept. If the connection fails live, the saved output is your safety net.

**First time?** Do the one-time setup in `../README.md` (shared environment, `demos/.env` with `GOOGLE_API_KEY`).

In [7]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1. Setup and the schema

The helpdesk schema is the same shape as the booking schema from the lecture: only the fields change. Each field has a `description` that tells the model what to look for, and to use `null` when the user does not say.

In [8]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Notebook tools choose different working folders. Find the one shared
# demos/.env whether this notebook was opened from the repo, demos, or here.
env_candidates = (Path.cwd() / ".env", Path.cwd() / "demos/.env", Path.cwd().parent / ".env")
demo_env = next((path for path in env_candidates if path.is_file()), None)
if demo_env is None:
    raise FileNotFoundError("Create demos/.env from demos/.env.example before continuing.")
load_dotenv(demo_env)

# Same default as every other module's demo; change it in demos/.env if needed.
GENERATION_MODEL = os.getenv("GENERATION_MODEL", "gemini-3.1-flash-lite")

# The rules for the reply. Each field may be null, and every key is required.
SCHEMA = {
    "type": "object",
    "properties": {
        "issue_category": {
            "type": ["string", "null"],
            "enum": ["billing", "access", "outage", None],
            "description": "What kind of problem it is. Use null if the user does not say.",
        },
        "urgency": {
            "type": ["string", "null"],
            "enum": ["low", "medium", "high", None],
            "description": "How soon this must be fixed. Use null if the user does not say.",
        },
        "affected_system": {
            "type": ["string", "null"],
            "description": "Which internal tool is broken, for example VPN. Use null if the user does not say.",
        },
    },
    "required": ["issue_category", "urgency", "affected_system"],
}

## 2. The call, and the validation

`extract` makes the same kind of call as the lecture: a prompt, plus the schema as the response format. `validate` is **our own** check of the reply. The API already enforces the shape, but we check anyway, because code must never trust a reply it did not verify (and because values need rules the schema cannot express).

In [9]:
def extract(sentence):
    """Ask the model to turn one helpdesk message into a ticket, as JSON that
    follows SCHEMA. Returns the reply as a Python dictionary."""
    client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    prompt = (
        "Turn this helpdesk message into a ticket.\n"
        f'Message: "{sentence}"'
    )
    interaction = client.interactions.create(
        model=GENERATION_MODEL,
        input=prompt,
        response_format={"type": "text", "mime_type": "application/json", "schema": SCHEMA},
    )
    return json.loads(interaction.output_text)


def validate(ticket):
    """Check a reply against the schema rules. Returns a list of problems;
    an empty list means the shape is valid."""
    problems = []
    props = SCHEMA["properties"]
    for key in SCHEMA["required"]:
        if key not in ticket:
            problems.append(f"missing required key: {key}")
    for key, value in ticket.items():
        if key not in props:
            continue  # JSON Schema allows extra keys unless the schema forbids them
        allowed = props[key].get("enum")
        if allowed is not None and value not in allowed:
            problems.append(f"{key} = {value!r} is not one of {allowed}")
        if value is not None and not isinstance(value, str):
            problems.append(f"{key} must be text or null")
    return problems

## 3. The test sentences

The three sentences from the slide. The **expected** column is what we *hope* for. The live run shows what the model really returns.

In [10]:
TESTS = [
    {
        "sentence": "The VPN is down for the whole office. Please fix it now.",
        "expected": {"issue_category": "outage", "urgency": "high", "affected_system": "VPN"},
        "tests": "the easy case: everything is stated",
    },
    {
        "sentence": "I cannot log in.",
        "expected": {"issue_category": "access", "urgency": None, "affected_system": None},
        "tests": "missing details should become null, not a guess",
    },
    {
        "sentence": "It is not urgent, but the payroll system is down and nobody can be paid.",
        "expected": None,  # the sentence disagrees with itself: there is no single right answer
        "tests": "unclear input: the fallback you chose has to decide",
    },
]

## 4. Run and read: the three checks

In [11]:
def read_result(test):
    """Run one test sentence and print the three checks, one line each."""
    print("SENTENCE:", test["sentence"])
    print("  tests: ", test["tests"])
    ticket = extract(test["sentence"])
    print("  reply: ", json.dumps(ticket))

    problems = validate(ticket)
    print("  check 1, valid shape?      ", "yes" if not problems else "NO: " + "; ".join(problems))

    if test["expected"] is None:
        print("  check 2, right values?      no single right answer here: this is the fallback's job")
    else:
        same = {k: ticket.get(k) for k in test["expected"]} == test["expected"]
        print("  check 2, right values?     ", "matches what we hoped for" if same else f"DIFFERENT. we hoped for {test['expected']}")

    unsure = [k for k, v in (test["expected"] or {}).items() if v is None]
    if unsure:
        guessed = [k for k in unsure if ticket.get(k) is not None]
        print("  check 3, null when unsure? ", "yes" if not guessed else f"NO, it guessed: {guessed}")
    else:
        print("  check 3, null when unsure?  not tested by this sentence")
    print()
    return ticket


for test in TESTS:
    read_result(test)

SENTENCE: The VPN is down for the whole office. Please fix it now.
  tests:  the easy case: everything is stated
  reply:  {"issue_category": "outage", "urgency": "high", "affected_system": "VPN"}
  check 1, valid shape?       yes
  check 2, right values?      matches what we hoped for
  check 3, null when unsure?  not tested by this sentence

SENTENCE: I cannot log in.
  tests:  missing details should become null, not a guess
  reply:  {"issue_category": "access", "urgency": null, "affected_system": null}
  check 1, valid shape?       yes
  check 2, right values?      matches what we hoped for
  check 3, null when unsure?  yes

SENTENCE: It is not urgent, but the payroll system is down and nobody can be paid.
  tests:  unclear input: the fallback you chose has to decide
  reply:  {"issue_category": "outage", "urgency": "low", "affected_system": "payroll"}
  check 1, valid shape?       yes
  check 2, right values?      no single right answer here: this is the fallback's job
  check 3, 

## 5. The fallback, and one rule of our own

The third sentence has no single right answer. A model may quietly choose one reading (for example `"low"`, because the message says "not urgent") and the reply is still valid JSON, so the schema check cannot notice. Two protections run here: the **fallback** (what to do when a value is missing or the reply fails validation) and **a rule of our own** (a payroll outage that nobody can fix is not "low", so we ask). Change `FALLBACK` to `"ask"`, `"default"` or `"escalate"` and run the cell again to show the three strategies from the slide. The domain decides which one is right: here a wrong urgency wastes a technician's morning, so `"ask"` is the safest.

In [12]:
FALLBACK = "ask"   # try "default" and "escalate" too


def handle(sentence):
    """Turn a message into a ticket, or, if a value is missing or the reply
    fails validation, run the chosen fallback instead. Never fails silently."""
    ticket = extract(sentence)
    if validate(ticket):
        return {"action": "retry", "note": "the reply failed validation, call again once"}
    # A rule of our own: the schema cannot know that an outage marked "low" is suspicious.
    if ticket["issue_category"] == "outage" and ticket["urgency"] == "low":
        return {"action": "ask", "say": "You said this is not urgent, but it is an outage. How urgent is it: low, medium or high?"}
    if ticket["urgency"] is None:
        if FALLBACK == "ask":
            return {"action": "ask", "say": "How urgent is this: low, medium or high?"}
        if FALLBACK == "default":
            return {"action": "file", "ticket": {**ticket, "urgency": "medium"}, "note": "urgency assumed"}
        return {"action": "escalate", "say": "I cannot file this. I am passing you to a person."}
    return {"action": "file", "ticket": ticket}


for test in TESTS:
    print(test["sentence"])
    print("  ->", handle(test["sentence"]), "\n")

The VPN is down for the whole office. Please fix it now.
  -> {'action': 'file', 'ticket': {'issue_category': 'outage', 'urgency': 'high', 'affected_system': 'VPN'}} 

I cannot log in.
  -> {'action': 'ask', 'say': 'How urgent is this: low, medium or high?'} 

It is not urgent, but the payroll system is down and nobody can be paid.
  -> {'action': 'ask', 'say': 'You said this is not urgent, but it is an outage. How urgent is it: low, medium or high?'} 

